# Notebook 00 — MRL Foundations

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Modulo 30 defines eight admissible residue lanes before prime classification:

```text
01 → 07 → 11 → 13 → 17 → 19 → 23 → 29
```

Constraint view:

> Mod30 Residue Lanes (MRL) studies admissible residue classes as discrete constraint surfaces before labeling numbers as prime or composite.


## Goals

1. Detect repo root and create standard output directories.
2. Define modulo-30 admissible residue lanes.
3. Verify the eight residues coprime to 30.
4. Generate a compact residue-wheel table.
5. Export JSON and Markdown foundation reports.
6. Generate a first wheel figure.
7. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Define MRL residue spine

Modulo 30 removes numbers divisible by 2, 3, or 5. The remaining admissible positions are the eight residues coprime to 30.


In [ ]:
MODULUS = 30
ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]

def residue_lane(n: int, modulus: int = MODULUS) -> int:
    return n % modulus

def admissible_mod30(n: int) -> bool:
    return residue_lane(n) in ADMISSIBLE_RESIDUES

def gcd_admissible(r: int) -> bool:
    return math.gcd(r, MODULUS) == 1

computed_residues = [r for r in range(MODULUS) if gcd_admissible(r)]

print("Declared residues:", ADMISSIBLE_RESIDUES)
print("Computed residues:", computed_residues)
print("Match:", ADMISSIBLE_RESIDUES == computed_residues)


## Build residue-wheel table

In [ ]:
rows = []

for r in range(MODULUS):
    rows.append({
        "position": r,
        "is_admissible": r in ADMISSIBLE_RESIDUES,
        "gcd_with_30": math.gcd(r, MODULUS),
        "lane_label": f"{r:02d}" if r in ADMISSIBLE_RESIDUES else "",
    })

wheel_df = pd.DataFrame(rows)
admissible_df = wheel_df[wheel_df["is_admissible"]].copy()

wheel_csv = RESULTS_DIR / "notebook00_mod30_wheel_table.csv"
admissible_csv = RESULTS_DIR / "notebook00_admissible_residue_lanes.csv"

wheel_df.to_csv(wheel_csv, index=False)
admissible_df.to_csv(admissible_csv, index=False)

print(wheel_csv)
print(admissible_csv)

admissible_df


## Foundation summary

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "00_mrl_foundations",
    "modulus": MODULUS,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "admissible_count": len(ADMISSIBLE_RESIDUES),
    "residue_spine": "01 → 07 → 11 → 13 → 17 → 19 → 23 → 29",
    "constraint_view": "MRL studies admissible residue classes as discrete constraint surfaces before prime/composite labeling.",
    "verification": {
        "computed_by_gcd": computed_residues,
        "declared_matches_computed": ADMISSIBLE_RESIDUES == computed_residues,
    },
}

json_path = RESULTS_DIR / "notebook00_mrl_foundations.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figure: modulo-30 residue wheel

In [ ]:
theta = np.linspace(0, 2*np.pi, MODULUS, endpoint=False)
x = np.cos(theta)
y = np.sin(theta)

fig, ax = plt.subplots(figsize=(7, 7))

for r in range(MODULUS):
    if r in ADMISSIBLE_RESIDUES:
        ax.scatter(x[r], y[r], s=180)
        ax.text(1.15*x[r], 1.15*y[r], f"{r:02d}", ha="center", va="center", fontsize=11)
    else:
        ax.scatter(x[r], y[r], s=35, alpha=0.35)

circle = plt.Circle((0, 0), 1, fill=False, linewidth=1)
ax.add_artist(circle)

ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Notebook 00 — Mod30 Residue Wheel\nAdmissible lanes: 01, 07, 11, 13, 17, 19, 23, 29")

figure_path = FIGURES_DIR / "notebook00_mod30_residue_wheel.png"
plt.tight_layout()
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

print(figure_path)


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_00_mrl_foundations.md"

report = f"""# Report 00 — MRL Foundations

Modulo 30 defines eight admissible residue lanes before prime classification:

```text
01 → 07 → 11 → 13 → 17 → 19 → 23 → 29
```

Constraint view:

> MRL studies admissible residue classes as discrete constraint surfaces before prime/composite labeling.

## Generated outputs

- Wheel table CSV: <a href="../results/notebook00_mod30_wheel_table.csv">`results/notebook00_mod30_wheel_table.csv`</a>
- Admissible residue lanes CSV: <a href="../results/notebook00_admissible_residue_lanes.csv">`results/notebook00_admissible_residue_lanes.csv`</a>
- Foundation JSON: <a href="../results/notebook00_mrl_foundations.json">`results/notebook00_mrl_foundations.json`</a>
- Residue wheel figure: <a href="../figures/notebook00_mod30_residue_wheel.png">`figures/notebook00_mod30_residue_wheel.png`</a>

## Verification

- Modulus: `{MODULUS}`
- Declared admissible residues: `{ADMISSIBLE_RESIDUES}`
- Computed residues coprime to 30: `{computed_residues}`
- Declared matches computed: `{ADMISSIBLE_RESIDUES == computed_residues}`

## Admissible residue table

{admissible_df.to_markdown(index=False)}

## Interpretation

- Modulo 30 removes multiples of 2, 3, and 5.
- Eight positions remain as admissible residue lanes.
- These lanes form the base MRL substrate for later lane-specific notebooks.
- Notebook 01 can begin with lane 1 as anchor behavior and compare it against the full eight-lane wheel.

## Next step

Notebook 01 — Lane Residue 1 should generate lane-local counts, spacing, interval statistics, and comparison against all eight admissible lanes.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:4000])


## Render generated figure

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown("### `notebook00_mod30_residue_wheel.png`"))
display(Image(filename=str(figure_path)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook00_mrl_foundations_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook00_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook00_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_00_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook00_mrl_foundations_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook00_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_00_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
